# Practica 1 - Probabilidad y estadistica 
## Analisis de la evolucion poblacional mundial
### Actividad 1 — Identificación de la fuente 

Reconocer la procedencia del conjunto de datos antes de utilizarlo y documentar correctamente su fuente. 

La fuente original es **United Nations, World Population Prospects (2024)**. 

**Cita:** 

> UN, World Population Prospects (2024) – processed by Our World in Data. “Annual change in population – UN WPP” [dataset]. United Nations, “World Population Prospects”; United Nations, “World Population Prospects - Interim Update” [original data]. 

Preguntas 

Responde brevemente: 

1. ¿Quién produjo originalmente los datos? United Nations
2. ¿Qué organización los procesa para su utilización en Our World in Data? Our World in Data
3. ¿Qué representa la variable que analizarás? La variable principal, **"Annual change in population" (cambio anual de población)**, representa el **cambio *neto* anual de la población**, calculado como la diferencia entre la población al 1 de julio de años consecutivos. Refleja de forma combinada nacimientos, defunciones y migración, y se mide en **número de personas** (valor entero, no un porcentaje ni una tasa de crecimiento). La serie de estimaciones cubre 1951–2023; la columna adicional "Annual population change (Projected)" contiene las proyecciones a futuro (2024–2100) basadas en el escenario medio de la ONU.

In [1]:
import pandas as pd 
import numpy as np 

df= pd.read_csv("../data/raw/annual-population-growth/annual-population-growth.csv")

## Actividad 2 — Carga y exploración de datos

**Propósito:** comprobar que Python, Pandas y Jupyter funcionan correctamente y realizar una primera exploración del dataset.

Las bibliotecas (`pandas`, `numpy`) ya fueron importadas y el archivo CSV ya fue cargado en el DataFrame `df` en la celda anterior (puntos 2.1 y 2.2), por lo que no se repiten. A continuación se explora el DataFrame (punto 2.3).

**Primeras filas**

In [2]:
df.head()

,Entity,Code,Year,Annual change in population,Annual population change (Projected)
0,Afghanistan,AFG,1951,103163.0,NaN
1,Afghanistan,AFG,1952,108441.0,NaN
2,Afghanistan,AFG,1953,108919.0,NaN
3,Afghanistan,AFG,1954,111251.0,NaN
4,Afghanistan,AFG,1955,119027.0,NaN


**Últimas filas**

In [3]:
df.tail()

,Entity,Code,Year,Annual change in population,Annual population change (Projected)
38395,Zimbabwe,ZWE,2096,NaN,100752.0
38396,Zimbabwe,ZWE,2097,NaN,91796.0
38397,Zimbabwe,ZWE,2098,NaN,87605.0
38398,Zimbabwe,ZWE,2099,NaN,76670.0
38399,Zimbabwe,ZWE,2100,NaN,70019.0


**Dimensiones (filas, columnas)**

In [4]:
df.shape

(38400, 5)

**Nombres de columnas**

In [5]:
df.columns.tolist()

['Entity',
 'Code',
 'Year',
 'Annual change in population',
 'Annual population change (Projected)']

**Tipos de datos**

In [6]:
df.dtypes

Entity                                      str
Code                                        str
Year                                      int64
Annual change in population             float64
Annual population change (Projected)    float64
dtype: object

**Estadísticas descriptivas**

In [7]:
df.describe()

,Year,Annual change in population,Annual population change (Projected)
count,38400.000000,1.868800e+04,1.971200e+04
mean,2025.500000,2.063019e+06,8.375481e+05
std,43.300872,9.440859e+06,5.816745e+06
min,1951.000000,-3.315935e+06,-2.291035e+07
25%,1988.000000,1.781000e+03,-9.742750e+03
50%,2025.500000,4.689800e+04,9.350000e+01
75%,2063.000000,3.021128e+05,1.411160e+05
max,2100.000000,9.337137e+07,7.206736e+07


**Cantidad de valores faltantes**

In [8]:
df.isnull().sum()

Entity                                      0
Code                                     1200
Year                                        0
Annual change in population             19712
Annual population change (Projected)    18688
dtype: int64

**Cantidad de entidades**

In [9]:
df["Entity"].unique()

<StringArray>
[        'Afghanistan',         'Africa (UN)',             'Albania',
             'Algeria',      'American Samoa',       'Americas (UN)',
             'Andorra',              'Angola',            'Anguilla',
 'Antigua and Barbuda',
 ...
             'Vanuatu',             'Vatican',           'Venezuela',
             'Vietnam',   'Wallis and Futuna',      'Western Sahara',
               'World',               'Yemen',              'Zambia',
            'Zimbabwe']
Length: 256, dtype: str

In [10]:
df["Entity"].nunique()

256

### Preguntas de la Actividad 2

**1. ¿Cuántas filas y columnas tiene el dataset?**

Según `df.shape`, el dataset tiene **38 400 filas y 5 columnas**.

**2. ¿Qué tipo de datos contiene la variable `Year`?**

Según `df.dtypes`, la columna `Year` es de tipo **`int64`** (entero), ya que contiene el año como número entero.

**3. ¿Existen valores faltantes?**

**Sí, existen valores faltantes.** Según `df.isnull().sum()` se distribuyen así:

- `Entity`: 0
- `Code`: 1 200
- `Year`: 0
- `Annual change in population`: 19 712
- `Annual population change (Projected)`: 18 688

Los faltantes en las dos columnas de datos son esperables: cada fila corresponde a estimaciones históricas (1951–2023) **o** a proyecciones (2024–2100), por lo que la columna que no aplica queda vacía. Los 1 200 faltantes en `Code` corresponden a entidades sin código ISO (por ejemplo, regiones o agregados).

**4. ¿Cuántas entidades diferentes contiene el dataset?**

Según `df["Entity"].nunique()`, el dataset contiene **256 entidades diferentes** (países, regiones y agregados).

## Actividad 3 — Estadística descriptiva

**Propósito:** aplicar medidas estadísticas básicas a una variable poblacional.

Se selecciona un año del **periodo histórico** (1951–2023). En este caso se elige el año **2020** y se filtra el dataset para ese año. Las medidas se calculan sobre la variable **`Annual change in population`** (cambio *neto* anual de población, en personas).

In [11]:
anio = 2020
col = "Annual change in population"

# Filtramos las filas del año histórico seleccionado
hist = df[df["Year"] == anio].copy()
serie = hist[col]

hist.head()

,Entity,Code,Year,Annual change in population,Annual population change (Projected)
69,Afghanistan,AFG,2020,1212852.0,NaN
219,Africa (UN),UN_AFR,2020,32815675.0,NaN
369,Albania,ALB,2020,-13061.0,NaN
519,Algeria,DZA,2020,747543.0,NaN
669,American Samoa,ASM,2020,-451.0,NaN


**Medidas estadísticas (media, mediana, varianza, desviación estándar, mínimo y máximo)**

In [12]:
print("Año analizado :", anio)
print("Media         :", serie.mean())
print("Mediana       :", serie.median())
print("Varianza      :", serie.var())
print("Desv. estándar:", serie.std())
print("Mínimo        :", serie.min())
print("Máximo        :", serie.max())

Año analizado : 2020
Media         : 2124908.84765625
Mediana       : 34293.5
Varianza      : 90687335031103.64
Desv. estándar: 9522989.815761836
Mínimo        : -494016.0
Máximo        : 75707590.0


**Entidad con el valor mínimo y con el valor máximo**

In [13]:
ent_min = hist.loc[serie.idxmin(), "Entity"]
ent_max = hist.loc[serie.idxmax(), "Entity"]
print("Entidad con el MÍNIMO:", ent_min, "->", serie.min())
print("Entidad con el MÁXIMO:", ent_max, "->", serie.max())

Entidad con el MÍNIMO: Venezuela -> -494016.0
Entidad con el MÁXIMO: World -> 75707590.0


### Preguntas de la Actividad 3 (año 2020)

**1. ¿Cuál es el valor de la media?**

La media es **2 124 908.85 personas** (aproximadamente 2.12 millones).

**2. ¿Cuál es el valor de la mediana?**

La mediana es **34 293.5 personas**. Es mucho menor que la media porque unas pocas entidades con valores muy grandes (agregados como *World*) elevan el promedio.

**3. ¿Cuál es el valor de la desviación estándar?**

La desviación estándar es **9 522 989.82 personas** (la varianza es ≈ 9.07 × 10¹³), lo que indica una dispersión muy alta entre entidades.

**4. ¿Qué entidad presenta el valor mínimo y cuál el máximo?**

- **Mínimo:** **Venezuela**, con **−494 016 personas** (población que *disminuye* ese año).
- **Máximo:** **World** (Mundo), con **75 707 590 personas**.

> Nota: el dataset incluye, además de países, entidades agregadas (por ejemplo *World*, regiones y grupos de ingreso). Por eso el valor máximo corresponde al agregado *World* y no a un país individual.

## Actividad 4 — Comprobación del entorno

**Propósito:** comprobar que el notebook usa correctamente las herramientas configuradas para la práctica, mostrando las versiones de Python, Pandas y NumPy.

In [14]:
import sys

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]
Pandas: 3.0.5
NumPy: 2.5.2


**Verificación de versiones del entorno virtual (`.venv`) de la práctica**

Al ejecutar la celda anterior dentro del entorno virtual del proyecto (`D:\big data\primera\.venv`), las versiones deben corresponder a:

- **Python:** 3.13.13
- **Pandas:** 3.0.5
- **NumPy:** 2.5.2

Si las versiones que muestra la celda coinciden con estas, el notebook está usando correctamente el entorno virtual configurado para la práctica. Como paso final, se recomienda ejecutar **todas las celdas desde el inicio** (*Run All*) para comprobar que el análisis completo se reproduce sin errores.

In [15]:
df_2005_2020 = df[(df["Year"] >= 2005) & (df["Year"] <= 2020) & (df["Entity"] == "Mexico" )]
df_2005_2020.shape 

(16, 5)

In [16]:
df_ext_EA= df_2005_2020[["Entity", "Annual change in population"]]
df_ext_EA.head(3)


,Entity,Annual change in population
22254,Mexico,1417373.0
22255,Mexico,1442162.0
22256,Mexico,1520690.0


In [17]:
df_ext_EA.mean(numeric_only=True)

Annual change in population    1400307.625
dtype: float64

In [18]:
#Pais 1er apellido de 1983 a 2021
df_paist_1983_2021 = df[(df["Year"] >= 1983) & (df["Year"] <= 2021) & (df["Entity"].str.startswith("Taiwan") )]
df_paist_1983_2021.head


<bound method NDFrame.head of        Entity Code  Year  Annual change in population  \
33632  Taiwan  TWN  1983                     298643.0   
33633  Taiwan  TWN  1984                     284133.0   
33634  Taiwan  TWN  1985                     264933.0   
33635  Taiwan  TWN  1986                     234286.0   
33636  Taiwan  TWN  1987                     209839.0   
33637  Taiwan  TWN  1988                     207838.0   
33638  Taiwan  TWN  1989                     195255.0   
33639  Taiwan  TWN  1990                     175994.0   
33640  Taiwan  TWN  1991                     186845.0   
33641  Taiwan  TWN  1992                     200336.0   
33642  Taiwan  TWN  1993                     192590.0   
33643  Taiwan  TWN  1994                     182467.0   
33644  Taiwan  TWN  1995                     177352.0   
33645  Taiwan  TWN  1996                     174773.0   
33646  Taiwan  TWN  1997                     176062.0   
33647  Taiwan  TWN  1998                     163894.0   
3

In [19]:
solo_paises = df[df["Code"].notna() & (df["Code"].str.len() == 3)]
fila_max_pais = solo_paises.loc[solo_paises[col].idxmax()]
print("Máximo (solo países):", fila_max_pais["Entity"], fila_max_pais["Year"], fila_max_pais[col])

Máximo (solo países): China 1970 21168623.0


In [24]:
df.iloc[600:620,0:12]

,Entity,Code,Year,Annual change in population,Annual population change (Projected)
600,American Samoa,ASM,1951,380.0,NaN
601,American Samoa,ASM,1952,126.0,NaN
602,American Samoa,ASM,1953,115.0,NaN
603,American Samoa,ASM,1954,91.0,NaN
604,American Samoa,ASM,1955,95.0,NaN
605,American Samoa,ASM,1956,83.0,NaN
606,American Samoa,ASM,1957,23.0,NaN
607,American Samoa,ASM,1958,-33.0,NaN
608,American Samoa,ASM,1959,-19.0,NaN
609,American Samoa,ASM,1960,228.0,NaN


In [26]:
df.iloc[10000]

Entity                                  Equatorial Guinea
Code                                                  GNQ
Year                                                 2051
Annual change in population                           NaN
Annual population change (Projected)              46407.0
Name: 10000, dtype: object

In [27]:
df.loc[:,["Entity", "Year", "Annual change in population"]]

,Entity,Year,Annual change in population
0,Afghanistan,1951,103163.0
1,Afghanistan,1952,108441.0
2,Afghanistan,1953,108919.0
3,Afghanistan,1954,111251.0
4,Afghanistan,1955,119027.0
...,...,...,...
38395,Zimbabwe,2096,NaN
38396,Zimbabwe,2097,NaN
38397,Zimbabwe,2098,NaN
38398,Zimbabwe,2099,NaN


In [29]:
df.iloc[:, [0,2,3]]

,Entity,Year,Annual change in population
0,Afghanistan,1951,103163.0
1,Afghanistan,1952,108441.0
2,Afghanistan,1953,108919.0
3,Afghanistan,1954,111251.0
4,Afghanistan,1955,119027.0
...,...,...,...
38395,Zimbabwe,2096,NaN
38396,Zimbabwe,2097,NaN
38397,Zimbabwe,2098,NaN
38398,Zimbabwe,2099,NaN


In [30]:
df.loc[10000,["Entity", "Year", "Annual change in population"]]

Entity                         Equatorial Guinea
Year                                        2051
Annual change in population                  NaN
Name: 10000, dtype: object

In [31]:
df.iloc[10000, [0,2,3]]

Entity                         Equatorial Guinea
Year                                        2051
Annual change in population                  NaN
Name: 10000, dtype: object

In [36]:
df.iloc[(df["Year"] == 2020) & (df["Entity"] == "Mexico"), :]


,Entity,Code,Year,Annual change in population,Annual population change (Projected)
22269,Mexico,MEX,2020,1036075.0,NaN
